# DATA266 Lab 1 - Task 2: Yelp sentiment classification

This notebook trains the baseline, CNN, and GRU models using embeddings learned from scratch. A GPU runtime is recommended.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime before continuing'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/DATA266_Lab1_checkpoints/task2
!rm -rf /content/DATA266_Lab1_Denisha
!git clone https://github.com/denishatank12/GenAI.git /content/DATA266_Lab1_Denisha
%cd /content/DATA266_Lab1_Denisha
!pip install -q -r requirements.txt
!mkdir -p reproducibility/raw_logs reproducibility/manifests
!python reproducibility/prepare_data.py --task task2 --out .

In [ ]:
%cd /content/DATA266_Lab1_Denisha/task2_sentiment/member_denisha
!python src/analyze_data.py
!python -u src/train.py --config config.yaml --checkpoint-dir /content/drive/MyDrive/DATA266_Lab1_checkpoints/task2 --resume-latest 2>&1 | tee ../../reproducibility/raw_logs/task2_denisha.log

In [ ]:
import json
from pathlib import Path
print(Path('outputs/metrics_summary.csv').read_text())
print(Path('outputs/metrics_report.csv').read_text())
print(json.dumps(json.loads(Path('outputs/metrics.json').read_text()), indent=2))
!mkdir -p checkpoints ../../reproducibility/manifests
!cp -f /content/drive/MyDrive/DATA266_Lab1_checkpoints/task2/*.pt checkpoints/
!python ../../reproducibility/collect_manifest.py --task task2 --command 'python3 -u src/train.py --config config.yaml --resume-latest' --dataset 'Yelp Polarity via Hugging Face dataset preparation' --checkpoint checkpoints/baseline.pt --checkpoint checkpoints/experimental_cnn.pt --checkpoint checkpoints/experimental_gru.pt --out ../../reproducibility/manifests/task2_denisha.json
!python ../../reproducibility/validate_results.py --task task2 --root ../..

In [ ]:
!zip -r /content/task2_denisha_artifacts.zip outputs checkpoints ../../reproducibility/raw_logs/task2_denisha.log ../../reproducibility/manifests/task2_denisha.json
from google.colab import files
files.download('/content/task2_denisha_artifacts.zip')